<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_03_4_early_stop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>


# T81-558: Anwendungen tiefer neuronaler Netzwerke

**Modul 3: Einführung in PyTorch**

- Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
– Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).


# Modul 3 Material

- Teil 3.1: Einführung in Deep Learning und neuronale Netzwerke [[Video]](https://www.youtube.com/watch?v=d-rU5IuFqLs&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=d-rU5IuFqLs&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
- Teil 3.2: Einführung in PyTorch [[Video]](https://www.youtube.com/watch?v=Pf-rrhMolm0&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=Pf-rrhMolm0&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
- Teil 3.3: Kodieren eines Feature-Vektors für PyTorch Deep Learning [[Video]](https://www.youtube.com/watch?v=7SGPm2tIT58&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=7SGPm2tIT58&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
- **Teil 3.4: Frühzeitiges Stoppen und Netzwerkpersistenz** [[Video]](https://www.youtube.com/watch?v=lS0vvIWiahU&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=lS0vvIWiahU&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
- Teil 3.5: Sequenzen vs. Klassen in PyTorch [[Video]](https://www.youtube.com/watch?v=NOu8jMZx3LY&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=NOu8jMZx3LY&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)

# Google CoLab-Anweisungen

Der folgende Code stellt sicher, dass Google CoLab ausgeführt wird und ordnet bei Bedarf Google Drive zu. Wir initialisieren das PyTorch-Gerät auch entweder auf GPU/MPS (falls verfügbar) oder CPU.


In [1]:
import torch

try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Nutzen Sie eine GPU oder MPS (Apple), sofern verfügbar. (siehe Modul 3.2)
import torch

has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwendetes Gerät: {device}")

Note: not using Google CoLab
Using device: mps


# Teil 3.4: Frühzeitiges Stoppen und Netzwerkpersistenz

Dieses Modul befasst sich mit zwei wesentlichen Aspekten des Trainings neuronaler Netzwerke: Frühzeitiges Stoppen und Speichern/Laden von PyTorch-Netzwerken. Frühzeitiges Stoppen ist eine Technik, die hilft, Überanpassung zu verhindern und die Modellleistung zu optimieren, indem Validierungsverluste während des Trainings überwacht werden. Wir können unnötige Iterationen vermeiden und Rechenressourcen sparen, indem wir den Trainingsprozess stoppen, wenn der Validierungsverlust zunimmt. Darüber hinaus werden wir untersuchen, wie PyTorch-Netzwerke gespeichert und geladen werden, sodass wir trainierte Modelle speichern und für Vorhersagen oder weiteres Training wiederverwenden können. Wenn Sie diese Techniken verstehen, können Sie die Leistung Ihrer Modelle optimieren und Ihre trainierten Netzwerke effizient verwalten.

## Frühzeitiges Stoppen

Es kann schwierig sein, zu bestimmen, wie viele Epochen durchlaufen werden müssen, um ein neuronales Netzwerk zu trainieren. Wenn Sie das neuronale Netzwerk über zu viele Epochen trainieren, tritt Überanpassung auf, und das neuronale Netzwerk wird bei neuen Daten nicht gut funktionieren, obwohl es im Trainingsset eine gute Genauigkeit erreicht hat. Überanpassung tritt auf, wenn ein neuronales Netzwerk bis zu dem Punkt trainiert wird, an dem es beginnt, sich Dinge zu merken, anstatt zu verallgemeinern, wie in Abbildung 3.OVER dargestellt.

**Abbildung 3.OVER: Trainings- vs. Validierungsfehler bei Überanpassung**
![Training vs. Validation Error for Overfitting](https://raw.githubusercontent.com/jeffheaton/t81_558_deep_learning/master/images/class_3_training_val.png "Training vs. Validation Error for Overfitting")

Es ist wichtig, den ursprünglichen Datensatz in mehrere Datensätze zu segmentieren:

- **Trainingsset**
- **Validierungssatz**
- **Holdout-Set**

Sie können diese Mengen auf verschiedene Arten konstruieren. Die folgenden Programme demonstrieren einige davon.

Die erste Methode ist ein Trainings- und Validierungssatz. Wir verwenden die Trainingsdaten, um das neuronale Netzwerk zu trainieren, bis sich der Validierungssatz nicht mehr verbessert. Dabei wird versucht, an einem nahezu optimalen Trainingspunkt anzuhalten. Diese Methode liefert nur genaue „Out-of-Sample“-Vorhersagen für den Validierungssatz; dies sind normalerweise 20 % der Daten. Die Vorhersagen für die Trainingsdaten werden zu optimistisch sein, da dies die Daten waren, die wir zum Trainieren des neuronalen Netzwerks verwendet haben. Abbildung 3.VAL zeigt, wie wir den Datensatz aufteilen.

**Abbildung 3.VAL: Training mit einem Validierungssatz**
![Training with a Validation Set](https://raw.githubusercontent.com/jeffheaton/t81_558_deep_learning/master/images/class_1_train_val.png "Training with a Validation Set")


Da PyTorch keine integrierte Funktion zum frühzeitigen Stoppen enthält, müssen wir eine eigene definieren. Wir werden in diesem Kurs die folgende **EarlyStopping**-Klasse verwenden.

Wir können dem **EarlyStopping**-Objekt mehrere Parameter bereitstellen:

- **min_delta** Dieser Wert sollte klein gehalten werden; er gibt die Mindeständerung an, die als Verbesserung betrachtet werden sollte. Eine noch kleinere Einstellung wird wahrscheinlich keine großen Auswirkungen haben.
- **Geduld** Wie lange sollte das Training warten, bis sich der Validierungsfehler behebt?
- **restore_best_weights** Normalerweise sollten Sie dies auf „true“ setzen, da dadurch die Gewichte auf die Werte zurückgesetzt werden, die sie hatten, als der Validierungssatz am höchsten war.

Wir werden uns jetzt ein Beispiel dieser Klasse in Aktion ansehen.


In [2]:
import copy


class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_model = None
        self.best_loss = None
        self.counter = 0
        self.status = ""

    def __call__(self, model, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
        elif self.best_loss - val_loss >= self.min_delta:
            self.best_model = copy.deepcopy(model.state_dict())
            self.best_loss = val_loss
            self.counter = 0
            self.status = f"Improvement found, counter reset to {self.counter}"
        else:
            self.counter += 1
            self.status = f"No improvement in the last {self.counter} epochs"
            if self.counter >= self.patience:
                self.status = f"Early stopping triggered after {self.counter} epochs."
                if self.restore_best_weights:
                    model.load_state_dict(self.best_model)
                return True
        return False

### Vorzeitiges Stoppen mit Klassifizierung

Wir sehen uns nun ein Beispiel für ein Klassifizierungstraining mit frühzeitigem Stoppen an. Wir trainieren das neuronale Netzwerk, bis sich der Fehler im Validierungssatz nicht mehr verbessert.


In [3]:
import time

import numpy as np
import pandas as pd
import torch
import tqdm
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Labelcoder, StandardScaler
from torch import nn
from torch.autograd import Variable
from torch.utils.data import DataLoader, TensorDataset

# Legen Sie einen Zufallsstartwert für die Reproduzierbarkeit fest
np.random.seed(42)
torch.manual_seed(42)


def load_data():
    df = read_csv(
        "https://data.heatonresearch.com/data/t81-558/iris.csv", na_values=["NA", "?"]
    )

    le = Labelcoder()

    x = df[["sepal_l", "sepal_w", "petal_l", "petal_w"]].values
    y = le.fit_transform(df["species"])
    species = le.classes_

    # Aufteilung in Validierungs- und Trainingssets
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.25, random_state=42
    )

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)

    # Numpy zu Torch Tensor
    x_train = torch.tensor(x_train, device=device, dtype=torch.float32)
    y_train = torch.tensor(y_train, device=device, dtype=torch.long)

    x_test = torch.tensor(x_test, device=device, dtype=torch.float32)
    y_test = torch.tensor(y_test, device=device, dtype=torch.long)

    return x_train, x_test, y_train, y_test, species


x_train, x_test, y_train, y_test, species = load_data()

# Erstellen von Datasets
BATCH_SIZE = 16

dataset_train = TensorDataset(x_train, y_train)
dataloader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)

dataset_test = TensorDataset(x_test, y_test)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=True)

# Erstellen Sie ein Modell mit nn.Sequential
model = nn.Sequential(
    nn.Linear(x_train.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, len(species)),
    nn.LogSoftmax(dim=1),
)

model = torch.compile(model, backend="aot_eager").to(device)

loss_fn = nn.CrossEntropyLoss()  # Kreuzentropieverlust

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
es = EarlyStopping()

epoch = 0
done = False
while epoch < 1000 and not done:
    epoch += 1
    steps = list(enumerate(dataloader_train))
    pbar = tqdm.tqdm(steps)
    model.train()
    for i, (x_batch, y_batch) in pbar:
        y_batch_pred = model(x_batch.to(device))
        loss = loss_fn(y_batch_pred, y_batch.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss, current = loss.item(), (i + 1) * len(x_batch)
        if i == len(steps) - 1:
            model.eval()
            pred = model(x_test)
            vloss = loss_fn(pred, y_test)
            if es(model, vloss):
                done = True
            pbar.set_description(
                f"Epoch: {epoch}, tloss: {loss}, vloss: {vloss:>7f}, {es.status}"
            )
        else:
            pbar.set_description(f"Epoch: {epoch}, tloss {loss:}")

Epoch: 1, tloss 0.7705620527267456:  14%|█▎       | 1/7 [00:02<00:12,  2.07s/it]/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/autograd/__init__.py:394: UserWarning: Error detected in LinearBackward0. Traceback of forward call that caused the error:
  File "/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/nn/modules/container.py", line 215, in forward
    input = module(input)
 (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1702400227158/work/torch/csrc/autograd/python_anomaly_mode.cpp:119.)
  result = Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[2024-01-07 06:28:49,617] [0/1] torch._dynamo.exc: [WARNING] Backend compiler failed with a fake tensor exception at 
[2024-01-07 06:28:49,617] [0/1] torch._dynamo.exc: [WARNING]   File "/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/nn/modules/container.py", line 216, in forward
[2024-01-07 06:28:49,617]

In [4]:
pred = model(x_test)
vloss = loss_fn(pred, y_test)
print(f"Verlust = {vloss}")

Loss = 0.008987338282167912


Wie Sie oben sehen können, haben wir nicht die Gesamtzahl der angeforderten Epochen verwendet. Das Training des neuronalen Netzwerks wurde beendet, sobald sich der Validierungssatz nicht mehr verbesserte.


In [5]:
from sklearn.metrics import accuracy_score

pred = model(x_test)
_, predict_classes = torch.max(pred, 1)
correct = accuracy_score(y_test.cpu(), predict_classes.cpu())
print(f"Genauigkeit: {korrekt}")

Accuracy: 1.0


### Vorzeitiges Stoppen mit Regression

Der folgende Code zeigt, wie wir Early Stopping auf ein Regressionsproblem anwenden können. Die Technik ähnelt dem Early Stopping-Code für die Klassifizierung, den wir gerade gesehen haben.


In [6]:
import time

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import tqdm
from sklearn import preprocessing
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch.autograd import Variable
from torch.utils.data import DataLoader, TensorDataset

# Lesen Sie den MPG-Datensatz.
df = read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv", na_values=["NA", "?"]
)

cars = df["name"]

# Fehlende Werte behandeln
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())

# Pandas zu Numpy
x = df[
    [
        "cylinders",
        "displacement",
        "horsepower",
        "weight",
        "acceleration",
        "year",
        "origin",
    ]
].values
y = df["mpg"].values  # Rückschritt

# Aufteilung in Validierungs- und Trainingssets
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42
)

# Numpy zu Torch Tensor
x_train = torch.tensor(x_train, device=device, dtype=torch.float32)
y_train = torch.tensor(y_train, device=device, dtype=torch.float32)

x_test = torch.tensor(x_test, device=device, dtype=torch.float32)
y_test = torch.tensor(y_test, device=device, dtype=torch.float32)


# Erstellen von Datasets
BATCH_SIZE = 16

dataset_train = TensorDataset(x_train, y_train)
dataloader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)

dataset_test = TensorDataset(x_test, y_test)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=True)


# Modell erstellen

model = nn.Sequential(
    nn.Linear(x_train.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1),
)

model = torch.compile(model, backend="aot_eager").to(device)

# Definieren Sie die Verlustfunktion für die Regression
loss_fn = nn.MSELoss()

# Definieren Sie den Optimierer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

es = EarlyStopping()

epoch = 0
done = False
while epoch < 1000 and not done:
    epoch += 1
    steps = list(enumerate(dataloader_train))
    pbar = tqdm.tqdm(steps)
    model.train()
    for i, (x_batch, y_batch) in pbar:
        y_batch_pred = model(x_batch).flatten()  #
        loss = loss_fn(y_batch_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss, current = loss.item(), (i + 1) * len(x_batch)
        if i == len(steps) - 1:
            model.eval()
            pred = model(x_test).flatten()
            vloss = loss_fn(pred, y_test)
            if es(model, vloss):
                done = True
            pbar.set_description(
                f"Epoch: {epoch}, tloss: {loss}, vloss: {vloss:>7f}, EStop:[{es.status}]"
            )
        else:
            pbar.set_description(f"Epoch: {epoch}, tloss {loss:}")

Epoch: 1, tloss: 261.3401794433594, vloss: 672.387024, EStop:[]: 100%|█| 19/19 [
Epoch: 2, tloss: 189.85874938964844, vloss: 192.848831, EStop:[Improvement found
Epoch: 3, tloss: 206.134521484375, vloss: 180.863342, EStop:[Improvement found, 
Epoch: 4, tloss: 189.10606384277344, vloss: 174.722321, EStop:[Improvement found
Epoch: 5, tloss: 200.2677459716797, vloss: 170.232376, EStop:[Improvement found,
Epoch: 6, tloss: 180.11239624023438, vloss: 165.325546, EStop:[Improvement found
Epoch: 7, tloss: 122.46980285644531, vloss: 152.626953, EStop:[Improvement found
Epoch: 8, tloss: 142.1746826171875, vloss: 144.341232, EStop:[Improvement found,
Epoch: 9, tloss: 67.58550262451172, vloss: 133.457535, EStop:[Improvement found,
Epoch: 10, tloss: 75.73149871826172, vloss: 118.177559, EStop:[Improvement found
Epoch: 11, tloss: 92.48848724365234, vloss: 126.596382, EStop:[No improvement in
Epoch: 12, tloss: 73.3027572631836, vloss: 86.829872, EStop:[Improvement found, 
Epoch: 13, tloss: 100.019737

Abschließend werten wir den Fehler aus.


In [7]:
from sklearn import metrics

# Messen Sie den RMSE-Fehler. RMSE wird häufig bei Regressionen eingesetzt.
pred = model(x_test)
score = torch.sqrt(torch.nn.functional.mse_loss(pred.flatten(), y_test))
print(f"Endergebnis (RMSE): {score}")

Final score (RMSE): 2.8826305866241455


In [8]:
y_test

tensor([33.0000, 28.0000, 19.0000, 13.0000, 14.0000, 27.0000, 24.0000, 13.0000,
        17.0000, 21.0000, 15.0000, 38.0000, 26.0000, 15.0000, 25.0000, 12.0000,
        31.0000, 17.0000, 16.0000, 31.0000, 22.0000, 22.0000, 22.0000, 33.5000,
        18.0000, 44.0000, 26.0000, 24.5000, 18.1000, 12.0000, 27.0000, 36.0000,
        23.0000, 24.0000, 37.2000, 16.0000, 21.0000, 19.2000, 16.0000, 29.0000,
        26.8000, 27.0000, 18.0000, 10.0000, 23.0000, 36.0000, 26.0000, 25.0000,
        25.0000, 25.0000, 22.0000, 34.1000, 32.4000, 13.0000, 23.5000, 14.0000,
        18.5000, 29.8000, 28.0000, 19.0000, 11.0000, 33.0000, 23.0000, 21.0000,
        23.0000, 25.0000, 23.8000, 34.4000, 24.5000, 13.0000, 34.7000, 14.0000,
        15.0000, 18.0000, 25.0000, 19.9000, 17.5000, 28.0000, 29.0000, 17.0000,
        16.0000, 27.0000, 37.0000, 36.1000, 23.0000, 14.0000, 32.8000, 29.9000,
        20.0000, 12.0000, 15.5000, 23.7000, 24.0000, 36.0000, 19.0000, 38.0000,
        29.0000, 21.5000, 27.9000, 14.00

## Speichern und Laden eines PyTorch-Neuralnetzwerks

Das Anpassen/Trainieren komplexer neuronaler Netzwerke dauert lange. Es ist hilfreich, diese neuronalen Netzwerke speichern zu können, damit Sie sie später erneut laden können. Ein neu geladenes neuronales Netzwerk muss nicht erneut trainiert werden. PyTorch speichert neuronale Netzwerke normalerweise als [pickle](https://wiki.python.org/moin/UsingPickle)-Dateien. Der folgende Code trainiert ein neuronales Netzwerk, um den Kraftstoffverbrauch eines Autos vorherzusagen, und speichert das Modell.


In [9]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# Zur Reproduzierbarkeit
torch.manual_seed(0)
np.random.seed(0)

# Lesen Sie den MPG-Datensatz.
df = read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv", na_values=["NA", "?"]
)

# Fehlende Werte behandeln
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())

# Funktionen und Ziel auswählen
features = df[
    [
        "cylinders",
        "displacement",
        "horsepower",
        "weight",
        "acceleration",
        "year",
        "origin",
    ]
]
target = df["mpg"]

# Normalisieren Sie die Funktionen
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Konvertieren Sie Numpy- in PyTorch-Tensoren
features_tensor = torch.tensor(
    scaled_features, device=device, dtype=torch.float32)
target_tensor = torch.tensor(target.values, device=device, dtype=torch.float32)

# In TensorDataset konvertieren
dataset = TensorDataset(features_tensor, target_tensor)

# In DataLoader konvertieren
data_loader = DataLoader(dataset, batch_size=32)

# Definieren Sie das neuronale Netzwerk mit nn.Sequential
model = nn.Sequential(
    nn.Linear(features_tensor.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1),
).to(device)

# Definieren Sie die Verlustfunktion für die Regression
loss_fn = nn.MSELoss()

# Definieren Sie den Optimierer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Trainiere 1000 Epochen lang.
model.train()
for epoch in range(1000):
    for batch_features, batch_target in data_loader:
        optimizer.zero_grad()
        out = model(batch_features).flatten()
        loss = loss_fn(out, batch_target)
        loss.backward()
        optimizer.step()

    # Zeigt den Status alle 100 Epochen an.
    if epoch % 100 == 0:
print(f"Epoche {epoch}, Verlust: {loss.item()}")

model.eval()
pred = model(features_tensor)

# Messen Sie den RMSE-Fehler. RMSE wird häufig bei Regressionen eingesetzt.
score = torch.sqrt(torch.nn.functional.mse_loss(pred.flatten(), target_tensor))
print(f"Punktzahl vor dem Speichern (RMSE): {score}")
torch.save(model, "mpg.pkl")

Epoch 0, loss: 743.8233032226562
Epoch 100, loss: 14.932530403137207
Epoch 200, loss: 10.759376525878906
Epoch 300, loss: 8.70166301727295
Epoch 400, loss: 6.822720527648926
Epoch 500, loss: 5.033351421356201
Epoch 600, loss: 5.736409664154053
Epoch 700, loss: 3.7082600593566895
Epoch 800, loss: 6.8669209480285645
Epoch 900, loss: 5.820879936218262
Before save score (RMSE): 1.7334096431732178


Der folgende Code richtet ein neuronales Netzwerk ein und liest die Daten (für Vorhersagen), löscht jedoch nicht das Modellverzeichnis und passt das neuronale Netzwerk nicht an. Der Code lädt die Gewichte aus der vorherigen Anpassung. Jetzt laden wir das Netzwerk neu und führen eine weitere Vorhersage durch. Der RMSE sollte genau mit dem vorherigen übereinstimmen, wenn wir das neuronale Netzwerk korrekt gespeichert und neu geladen haben.


In [10]:
# Messen Sie den RMSE-Fehler für das belastete Netzwerk. RMSE wird häufig bei Regressionen eingesetzt.
model.eval()
pred = model(features_tensor)
score = torch.sqrt(torch.nn.functional.mse_loss(pred.flatten(), target_tensor))
print(f"Punktzahl vor dem Speichern (RMSE): {score}")
torch.save(model, "mpg.pkl")

Before save score (RMSE): 1.7334096431732178
